In [1]:
# ==============================================================================
# 0. INSTALL DEPENDENCIES (Run this at the very top of the cell)
# ==============================================================================
!pip install timm albumentations grad-cam --quiet

# ==============================================================================
# 1. IMPORTS & CONFIGURATION
# ==============================================================================
import os, cv2, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

class CFG:
    SEED            = 42
    IMG_SIZE        = 224
    BATCH_SIZE      = 64
    NUM_CLASSES     = 5
    FUSION_DIM      = 512
    NUM_HEADS       = 8
    DROPOUT         = 0.3
    
    # ⚠️ KAGGLE INPUT PATHS ⚠️
    NB1_DIR = Path('/kaggle/input/notebooks/soumyaranjansahoo33/cervixvit-data-preprocessing-eda')
    NB2_DIR = Path('/kaggle/input/notebooks/soumyaranjansahoo33/cervixvit-novel-architecture-training')
    
    WORK_DIR        = Path('/kaggle/working') # Where plots will be saved
    CLASS_NAMES     = ['Dyskeratotic', 'Koilocytotic', 'Metaplastic', 'Parabasal', 'Superficial_Intermediate']
    CLASS_SHORT     = ['Dyskerat.', 'Koiloc.', 'Metaplas.', 'Parabasal', 'Superf.']

# ==============================================================================
# 1.5 PATH FIX HELPER (Dynamic Search - Solves Kaggle absolute path corruption)
# ==============================================================================
_CACHED_DATASET_ROOT = None

def fix_image_path(img_path):
    global _CACHED_DATASET_ROOT
    
    # 1. If it works out of the box, return it
    if os.path.exists(img_path):
        return img_path
        
    # 2. Extract the target parts of the path
    parts = img_path.replace('\\', '/').split('/')
    class_idx = next((i for i, p in enumerate(parts) if p.lower().startswith('im_')), None)
    
    if class_idx is None:
        return img_path
        
    rel_parts = parts[class_idx:] # e.g., ['im_Dyskeratotic', 'im_Dyskeratotic', 'CROPPED', '164_19.bmp']
    target_file = rel_parts[-1]
    class_folder = rel_parts[0]

    # 3. Dynamically search Kaggle's input folder ONCE to find where the dataset actually lives
    if _CACHED_DATASET_ROOT is None:
        for root, dirs, _ in os.walk('/kaggle/input'):
            if root.count('/') > 5: continue # Prevent deep searching
            # If we find the class folders, we've found the dataset root
            if any(d.lower().startswith('im_') for d in dirs):
                _CACHED_DATASET_ROOT = root
                break
        if _CACHED_DATASET_ROOT is None:
            _CACHED_DATASET_ROOT = '/kaggle/input' # Fallback

    # 4. Try building the path using the dynamically found root
    cand1 = os.path.join(_CACHED_DATASET_ROOT, *rel_parts)
    if os.path.exists(cand1): return cand1
    
    # Handle variations in Kaggle's double-nesting (e.g., im_Class/CROPPED vs im_Class/im_Class/CROPPED)
    if len(rel_parts) > 1 and rel_parts[0] == rel_parts[1]:
        cand2 = os.path.join(_CACHED_DATASET_ROOT, rel_parts[0], *rel_parts[2:])
        if os.path.exists(cand2): return cand2

    # 5. Force-find the exact file inside the found class folder
    search_dir = os.path.join(_CACHED_DATASET_ROOT, class_folder)
    if os.path.exists(search_dir):
        for root, _, files in os.walk(search_dir):
            if target_file in files:
                return os.path.join(root, target_file)
                
    return img_path # Return broken path to trigger OpenCV error cleanly if completely missing
# ==============================================================================
# 2. ARCHITECTURE DEFINITIONS
# ==============================================================================
class CrossAttentionFusionModule(nn.Module):
    def __init__(self, dim_cnn=1536, dim_vit=768, out_dim=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.proj_cnn = nn.Sequential(nn.Linear(dim_cnn, out_dim), nn.LayerNorm(out_dim), nn.GELU())
        self.proj_vit = nn.Sequential(nn.Linear(dim_vit, out_dim), nn.LayerNorm(out_dim), nn.GELU())
        self.attn_cnn2vit = nn.MultiheadAttention(out_dim, num_heads, dropout=dropout, batch_first=True)
        self.attn_vit2cnn = nn.MultiheadAttention(out_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm_c   = nn.LayerNorm(out_dim)
        self.norm_v   = nn.LayerNorm(out_dim)
        self.gate     = nn.Parameter(torch.tensor(0.5))
        self.ffn      = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim * 4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(out_dim * 4, out_dim), nn.Dropout(dropout)
        )
        self.norm_out = nn.LayerNorm(out_dim)

    def forward(self, cnn_feat, vit_feat):
        q_c = self.proj_cnn(cnn_feat).unsqueeze(1)
        q_v = self.proj_vit(vit_feat).unsqueeze(1)
        c2v, _ = self.attn_cnn2vit(q_c, q_v, q_v)
        v2c, _ = self.attn_vit2cnn(q_v, q_c, q_c)
        c_rich = self.norm_c(q_c + c2v).squeeze(1)
        v_rich = self.norm_v(q_v + v2c).squeeze(1)
        g = torch.sigmoid(self.gate)
        merged = torch.cat([g * c_rich, (1-g) * v_rich], dim=-1)
        return self.norm_out(self.ffn(merged) + (c_rich + v_rich) / 2)

class DualPathCervixNet(nn.Module):
    def __init__(self, num_classes=5, fusion_dim=512, num_heads=8, dropout=0.3):
        super().__init__()
        self.cnn = timm.create_model('efficientnet_b3', pretrained=False, num_classes=0, global_pool='avg')
        self.vit = timm.create_model('swin_tiny_patch4_window7_224', pretrained=False, num_classes=0, global_pool='avg')
        self.cafm = CrossAttentionFusionModule(self.cnn.num_features, self.vit.num_features, fusion_dim, num_heads)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(fusion_dim, 256), nn.GELU(),
            nn.Dropout(dropout/2), nn.Linear(256, num_classes)
        )
        self.proj = nn.Sequential(nn.Linear(fusion_dim, 256), nn.GELU(), nn.Linear(256, 128))

    def forward(self, x, return_proj=False):
        fused  = self.cafm(self.cnn(x), self.vit(x))
        logits = self.classifier(fused)
        if return_proj:
            return logits, self.proj(fused)
        return logits

# ==============================================================================
# 3. LOAD DATA & CHECKPOINT
# ==============================================================================
print('Loading Checkpoint...')
ckpt = torch.load(CFG.NB2_DIR / 'best_model.pth', map_location=DEVICE, weights_only=False)
model = DualPathCervixNet(num_classes=CFG.NUM_CLASSES, fusion_dim=CFG.FUSION_DIM, num_heads=CFG.NUM_HEADS, dropout=CFG.DROPOUT).to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

class CervixDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        valid_path = fix_image_path(row['path']) # <--- PATH FIX APPLIED
        
        img_bgr = cv2.imread(valid_path)
        if img_bgr is None:
            raise FileNotFoundError(f"OpenCV failed to read: {valid_path}")
            
        image = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)['image']
        return image, int(row['label'])

tf = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

df_test = pd.read_csv(CFG.NB1_DIR / 'test.csv')
test_dl = DataLoader(CervixDataset(df_test, tf), CFG.BATCH_SIZE, shuffle=False, num_workers=2)

# ==============================================================================
# 4. INFERENCE & METRICS
# ==============================================================================
print(f'\nRunning Inference on {len(df_test)} test images...')
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_dl:
        imgs = imgs.to(DEVICE)
        with autocast():
            logits = model(imgs)
        probs = torch.softmax(logits, dim=-1)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

acc = accuracy_score(all_labels, all_preds)
f1_mac = f1_score(all_labels, all_preds, average='macro')
y_bin = label_binarize(all_labels, classes=list(range(CFG.NUM_CLASSES)))
auc = roc_auc_score(y_bin, all_probs, multi_class='ovr', average='macro')

print('\n=======================================================')
print('   DualPathCervixNet — Test Set Results')
print('=======================================================')
print(f'   Accuracy        : {acc:.4f}  ({acc*100:.2f}%)')
print(f'   Macro F1        : {f1_mac:.4f}')
print(f'   Macro AUC (OvR) : {auc:.4f}')
print('=======================================================\n')
print(classification_report(all_labels, all_preds, target_names=CFG.CLASS_SHORT, digits=4))

# ==============================================================================
# 5. GENERATE PLOTS (Confusion Matrix & ROC)
# ==============================================================================
print('\nGenerating Plots...')
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', xticklabels=CFG.CLASS_SHORT, yticklabels=CFG.CLASS_SHORT, linewidths=0.5, linecolor='white', cbar_kws={'label': 'Recall (%)'}, ax=ax)
ax.set_title('DualPathCervixNet — Confusion matrix (test set)', fontweight='bold')
plt.tight_layout()
plt.savefig(CFG.WORK_DIR / 'confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.close()

# ROC Curves
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#4E79A7','#F28E2B','#59A14F','#E15759','#76B7B2']
for i, (cls, col) in enumerate(zip(CFG.CLASS_SHORT, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    auc_i = roc_auc_score(y_bin[:, i], all_probs[:, i])
    ax.plot(fpr, tpr, label=f'{cls} (AUC={auc_i:.3f})', color=col, lw=1.8)
ax.plot([0,1],[0,1],'--', color='gray', lw=1, label='Random')
ax.set_title('ROC curves — DualPathCervixNet', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(CFG.WORK_DIR / 'roc_curves.png', bbox_inches='tight', dpi=150)
plt.close()

# ==============================================================================
# 6. GRAD-CAM VISUALIZATION
# ==============================================================================
def get_gradcam_heatmap(model, img_tensor, target_layer):
    cam = GradCAM(model=model, target_layers=[target_layer])
    return cam(input_tensor=img_tensor, targets=None)[0]

target_layer = model.cnn.blocks[-1][-1]
fig, axes = plt.subplots(5, 3, figsize=(10, 17))
col_titles = ['Original cell', 'GradCAM (EfficientNet)', 'Fused explanation']
for j, ct in enumerate(col_titles): axes[0, j].set_title(ct, fontweight='bold', fontsize=10)

for row, cls_idx in enumerate(range(CFG.NUM_CLASSES)):
    candidates = np.where((all_labels == cls_idx) & (all_preds == cls_idx))[0]
    if len(candidates) == 0: candidates = np.where(all_labels == cls_idx)[0]
    idx = candidates[0]
    
    valid_path = fix_image_path(df_test.iloc[idx]['path']) # <--- PATH FIX APPLIED
    img_bgr = cv2.imread(valid_path)
    
    img_rgb = cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), (CFG.IMG_SIZE, CFG.IMG_SIZE))
    img_01 = img_rgb.astype(np.float32) / 255.0
    img_norm = (img_01 - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
    img_tensor = torch.tensor(img_norm).permute(2,0,1).unsqueeze(0).float().to(DEVICE)
    
    try:
        heatmap = get_gradcam_heatmap(model, img_tensor, target_layer)
        cam_img = show_cam_on_image(img_01, heatmap, use_rgb=True)
    except:
        cam_img = img_rgb
        heatmap = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE), dtype=np.float32)

    hm_colored = plt.cm.jet(heatmap)[:, :, :3]
    fused_img = (0.6 * img_01 + 0.4 * hm_colored).clip(0, 1)

    axes[row, 0].imshow(img_rgb)
    axes[row, 0].set_ylabel(CFG.CLASS_SHORT[cls_idx], rotation=0, labelpad=60, fontsize=9, fontweight='bold', va='center')
    axes[row, 1].imshow(cam_img)
    axes[row, 2].imshow(fused_img)
    for j in range(3): axes[row, j].axis('off')

plt.suptitle('DualPathCervixNet — GradCAM explainability (test set)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(CFG.WORK_DIR / 'gradcam_figure.png', bbox_inches='tight', dpi=150)
plt.close()

# ==============================================================================
# 7. LEARNING CURVES FROM HISTORY
# ==============================================================================
with open(CFG.NB2_DIR / 'history.json') as f: history = json.load(f)
ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, history['train_loss'], label='Train')
axes[0].plot(ep, history['val_loss'], label='Val')
axes[0].set_title('CWCFL total loss'); axes[0].legend()
axes[1].plot(ep, [v*100 for v in history['val_acc']], label='Val Acc')
axes[1].plot(ep, [v*100 for v in history['val_f1']], label='Val F1')
axes[1].set_title('Validation metrics'); axes[1].legend()
plt.suptitle('DualPathCervixNet — Training history', fontweight='bold')
plt.tight_layout()
plt.savefig(CFG.WORK_DIR / 'training_history.png', bbox_inches='tight', dpi=150)
plt.close()

print('\n✅ All processes complete! Visualizations saved to /kaggle/working/')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 58.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Device: cuda
Loading Checkpoint...

Running Inference on 608 test images...

   DualPathCervixNet — Test Set Results
   Accuracy        : 0.9868  (98.68%)
   Macro F1        : 0.9869
   Macro AUC (OvR) : 0.9988

              precision    recall  f1-score   support

   Dyskerat.     0.9677    0.9836    0.9756       122
     Koiloc.     0.9835    0.9597    0.9714       124
   Metaplas.     0.9917    1.0000    0.9958       119
   Parabasal     1.0000    0.9915    0.9957       118
     Superf.     0.9921    1.0000    0.9960       125

    accuracy                         0.9868       608
   macro avg     0.9870    0.9870    0.9869       608
weighted avg     0.9869    0.9868    0.9868       608


Generating Plots...

✅ All processes complete! Visualizations saved to /kaggle/worki

In [2]:
# ==============================================================================
# 8. Q1 JOURNAL PUBLICATION PLOTS 
# ==============================================================================
import matplotlib.patches as mpatches
from sklearn.manifold import TSNE
import random

print('\nGenerating Publication-Ready Figures...')
CFG.FIG_DIR = CFG.WORK_DIR / 'paper_figures'
CFG.FIG_DIR.mkdir(parents=True, exist_ok=True)

# Helper to denormalize images for plotting
def denormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.permute(1, 2, 0).cpu().numpy()
    img = std * img + mean
    return np.clip(img, 0, 1)

# ------------------------------------------------------------------------------
# FIGURE A: t-SNE Feature Space Visualization (Proves CWCFL works)
# ------------------------------------------------------------------------------
print('Extracting features for t-SNE...')
features_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for imgs, labels in test_dl:
        imgs = imgs.to(DEVICE)
        # Extract features from the projection head (used in contrastive loss)
        _, proj_features = model(imgs, return_proj=True) 
        features_list.append(proj_features.cpu().numpy())
        labels_list.append(labels.numpy())

features_all = np.vstack(features_list)
labels_all = np.concatenate(labels_list)

print('Computing t-SNE embeddings (this takes a moment)...')
tsne = TSNE(n_components=2, perplexity=30, random_state=CFG.SEED)
tsne_results = tsne.fit_transform(features_all)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#4E79A7', '#F28E2B', '#59A14F', '#E15759', '#76B7B2']

for i, cls in enumerate(CFG.CLASS_SHORT):
    idx = (labels_all == i)
    ax.scatter(tsne_results[idx, 0], tsne_results[idx, 1], 
               c=colors[i], label=cls, alpha=0.7, edgecolors='w', s=50)

ax.set_title('t-SNE Feature Space (Projection Head)', fontweight='bold', fontsize=14)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(title="Cell Types", bbox_to_anchor=(1.05, 1), loc='upper left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(CFG.FIG_DIR / 'tsne_feature_space.png', bbox_inches='tight', dpi=300)
plt.close()

# ------------------------------------------------------------------------------
# FIGURE B: Original vs. Preprocessed/Augmented Images
# ------------------------------------------------------------------------------
print('Generating Preprocessing Grid...')
# Sample one image per class from the dataframe
sample_indices = [df_test[df_test['label'] == i].index[0] for i in range(CFG.NUM_CLASSES)]

fig, axes = plt.subplots(CFG.NUM_CLASSES, 2, figsize=(6, 12))
fig.suptitle('Raw Data vs. Preprocessed Input', fontweight='bold', fontsize=14, y=0.98)

for i, idx in enumerate(sample_indices):
    row = df_test.iloc[idx]
    
    # Use the path fix helper from earlier
    img_path = fix_image_path(row['path'])
    raw_img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    raw_img_resized = cv2.resize(raw_img, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    
    # Get the actual tensor fed to the model
    tensor_img, _ = test_dl.dataset[idx] 
    display_img = denormalize(tensor_img)
    
    # Plot Raw
    axes[i, 0].imshow(raw_img_resized)
    axes[i, 0].axis('off')
    if i == 0: axes[i, 0].set_title('Original Image', fontweight='bold')
    axes[i, 0].text(-0.1, 0.5, CFG.CLASS_SHORT[i], transform=axes[i, 0].transAxes, 
                    fontsize=11, fontweight='bold', va='center', ha='right', rotation=90)

    # Plot Preprocessed
    axes[i, 1].imshow(display_img)
    axes[i, 1].axis('off')
    if i == 0: axes[i, 1].set_title('Normalized Tensor', fontweight='bold')

plt.subplots_adjust(wspace=0.05, hspace=0.1)
plt.savefig(CFG.FIG_DIR / 'preprocessing_grid.png', bbox_inches='tight', dpi=300)
plt.close()

# ------------------------------------------------------------------------------
# FIGURE C: Qualitative Prediction Showcase (3x3 Grid)
# ------------------------------------------------------------------------------
print('Generating Prediction Grid...')
# Pick 9 random test images
random.seed(CFG.SEED)
grid_indices = random.sample(range(len(test_dl.dataset)), 9)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle('DualPathCervixNet: Qualitative Predictions', fontweight='bold', fontsize=16, y=0.95)

for i, idx in enumerate(grid_indices):
    r, c = divmod(i, 3)
    tensor_img, true_label = test_dl.dataset[idx]
    
    # Forward pass for single image
    model.eval()
    with torch.no_grad():
        img_input = tensor_img.unsqueeze(0).to(DEVICE)
        logits = model(img_input)
        pred_label = logits.argmax(1).item()
        conf = torch.softmax(logits, dim=-1)[0, pred_label].item()

    display_img = denormalize(tensor_img)
    axes[r, c].imshow(display_img)
    axes[r, c].axis('off')
    
    true_name = CFG.CLASS_SHORT[true_label]
    pred_name = CFG.CLASS_SHORT[pred_label]
    
    # Green text for correct, Red for incorrect
    color = '#2ca02c' if true_label == pred_label else '#d62728'
    title_text = f"True: {true_name}\nPred: {pred_name} ({conf:.2f})"
    axes[r, c].set_title(title_text, color=color, fontsize=10, fontweight='bold')

plt.tight_layout(rect=[0, 0.03, 1, 0.93])
plt.savefig(CFG.FIG_DIR / 'prediction_grid.png', bbox_inches='tight', dpi=300)
plt.close()

print(f'\n✅ High-res publication plots saved to: {CFG.FIG_DIR}')


Generating Publication-Ready Figures...
Extracting features for t-SNE...
Computing t-SNE embeddings (this takes a moment)...
Generating Preprocessing Grid...
Generating Prediction Grid...

✅ High-res publication plots saved to: /kaggle/working/paper_figures
